In [0]:
%run ../00-common/config

In [0]:
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Imputer,
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [0]:
orders = spark.table(
    f"{catalog_name}.{silver_schema}.orders"
)

order_items = spark.table(
    f"{catalog_name}.{silver_schema}.order_items"
)

products = spark.table(
    f"{catalog_name}.{silver_schema}.products"
)

print("Orders:", orders.count())
print("Order items:", order_items.count())
print("Products:", products.count())

In [0]:
required_orders = {
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "delivery_variance_days"
}

required_items = {
    "order_id",
    "product_id",
    "seller_id",
    "price",
    "freight_value"
}

required_products = {
    "product_id",
    "product_weight_g"
}

assert required_orders.issubset(set(orders.columns)), \
    f"Missing orders columns: {required_orders - set(orders.columns)}"

assert required_items.issubset(set(order_items.columns)), \
    f"Missing order_items columns: {required_items - set(order_items.columns)}"

assert required_products.issubset(set(products.columns)), \
    f"Missing products columns: {required_products - set(products.columns)}"

print("✓ Required ML source columns are available")

In [0]:
item_features = (
    order_items
    .join(
        products.select(
            "product_id",
            "product_weight_g"
        ),
        on="product_id",
        how="left"
    )
    .groupBy("order_id")
    .agg(
        F.count("*").alias("item_count"),

        F.sum("price")
         .alias("total_item_value"),

        F.sum("freight_value")
         .alias("total_freight"),

        F.avg("product_weight_g")
         .alias("avg_product_weight_g"),

        F.countDistinct("seller_id")
         .alias("seller_count"),

        F.countDistinct("product_id")
         .alias("product_count")
    )
)

print("Orders with item features:", item_features.count())

In [0]:
ml_data = (
    orders
    .filter(
        (F.col("order_status") == "delivered")
        & F.col("delivery_variance_days").isNotNull()
        & F.col("order_purchase_timestamp").isNotNull()
        & F.col("order_estimated_delivery_date").isNotNull()
    )
    .select(
        "order_id",
        "customer_id",
        "order_purchase_timestamp",
        "order_estimated_delivery_date",
        F.col("delivery_variance_days")
            .cast("double")
            .alias("label")
    )
    .join(
        customers.select(
            "customer_id",
            "customer_state"
        ),
        on="customer_id",
        how="left"
    )
    .join(
        item_features,
        on="order_id",
        how="inner"
    )
    .withColumn(
        "promised_delivery_days",
        F.datediff(
            F.to_date("order_estimated_delivery_date"),
            F.to_date("order_purchase_timestamp")
        ).cast("double")
    )
    .withColumn(
        "purchase_month",
        F.month("order_purchase_timestamp").cast("double")
    )
    .withColumn(
        "purchase_day_of_week",
        F.dayofweek("order_purchase_timestamp").cast("double")
    )
    .withColumn(
        "date_key",
        F.date_format(
            "order_purchase_timestamp",
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "purchase_epoch",
        F.col("order_purchase_timestamp").cast("long")
    )
)

print("ML dataset rows:", ml_data.count())

ml_data.select(
    "order_id",
    "label",
    "customer_state",
    "promised_delivery_days",
    "item_count",
    "total_item_value",
    "total_freight",
    "avg_product_weight_g",
    "seller_count",
    "product_count",
    "purchase_month",
    "purchase_day_of_week"
).show(10, truncate=False)

In [0]:
ml_data.select(
    F.count("*").alias("orders"),
    F.avg("label").alias("avg_delivery_variance"),
    F.min("label").alias("min_delivery_variance"),
    F.max("label").alias("max_delivery_variance")
).show()

In [0]:
cutoff_epoch = ml_data.approxQuantile(
    "purchase_epoch",
    [0.80],
    0.001
)[0]

train_df = ml_data.filter(
    F.col("purchase_epoch") <= F.lit(cutoff_epoch)
)

test_df = ml_data.filter(
    F.col("purchase_epoch") > F.lit(cutoff_epoch)
)

train_count = train_df.count()
test_count = test_df.count()
total_count = train_count + test_count

print("Train rows:", train_count)
print("Test rows:", test_count)

print(
    f"Train %: {train_count / total_count * 100:.1f}%"
)

print(
    f"Test %: {test_count / total_count * 100:.1f}%"
)

print(
    "Chronological cutoff:",
    train_df.agg(
        F.max("order_purchase_timestamp")
    ).first()[0]
)

In [0]:
numeric_feature_cols = [
    "item_count",
    "total_item_value",
    "total_freight",
    "avg_product_weight_g",
    "seller_count",
    "product_count",
    "promised_delivery_days",
    "purchase_month",
    "purchase_day_of_week"
]

imputed_cols = [
    f"{c}_imputed"
    for c in numeric_feature_cols
]

In [0]:
imputer = Imputer(
    inputCols=numeric_feature_cols,
    outputCols=imputed_cols,
    strategy="median"
)

state_indexer = StringIndexer(
    inputCol="customer_state",
    outputCol="customer_state_index",
    handleInvalid="keep"
)

state_encoder = OneHotEncoder(
    inputCols=["customer_state_index"],
    outputCols=["customer_state_encoded"]
)

assembler = VectorAssembler(
    inputCols=imputed_cols + [
        "customer_state_encoded"
    ],
    outputCol="features"
)

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="label",
    predictionCol="predicted_delivery_delay_days",
    numTrees=100,
    maxDepth=8,
    seed=42
)

ml_pipeline = Pipeline(
    stages=[
        imputer,
        state_indexer,
        state_encoder,
        assembler,
        rf
    ]
)

In [0]:
model = ml_pipeline.fit(train_df)

test_predictions = model.transform(test_df)

print("✓ Random Forest trained")

In [0]:
test_predictions = model.transform(test_df)

test_predictions.select(
    "order_id",
    "label",
    "predicted_delivery_delay_days"
).show(10, truncate=False)

In [0]:
baseline_delay = train_df.approxQuantile(
    "label",
    [0.5],
    0.001
)[0]

print(
    f"Training median delay: "
    f"{baseline_delay:.2f} days"
)

In [0]:
baseline_predictions = (
    test_df
    .withColumn(
        "baseline_prediction",
        F.lit(baseline_delay)
    )
)

In [0]:
rf_mae_evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="predicted_delivery_delay_days",
    metricName="mae"
)

rf_rmse_evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="predicted_delivery_delay_days",
    metricName="rmse"
)

rf_r2_evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="predicted_delivery_delay_days",
    metricName="r2"
)

baseline_mae_evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="baseline_prediction",
    metricName="mae"
)

baseline_rmse_evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="baseline_prediction",
    metricName="rmse"
)

rf_mae = rf_mae_evaluator.evaluate(test_predictions)
rf_rmse = rf_rmse_evaluator.evaluate(test_predictions)
rf_r2 = rf_r2_evaluator.evaluate(test_predictions)

baseline_mae = baseline_mae_evaluator.evaluate(
    baseline_predictions
)

baseline_rmse = baseline_rmse_evaluator.evaluate(
    baseline_predictions
)

print("=== BASELINE ===")
print(f"MAE:  {baseline_mae:.2f} days")
print(f"RMSE: {baseline_rmse:.2f} days")

print("\n=== RANDOM FOREST ===")
print(f"MAE:  {rf_mae:.2f} days")
print(f"RMSE: {rf_rmse:.2f} days")
print(f"R²:   {rf_r2:.3f}")

mae_improvement = (
    (baseline_mae - rf_mae)
    / baseline_mae
    * 100
)

print(
    f"\nMAE improvement vs baseline: "
    f"{mae_improvement:.1f}%"
)

In [0]:
train_predictions = (
    model
    .transform(train_df)
    .withColumn(
        "dataset_split",
        F.lit("train")
    )
)

test_predictions = (
    model
    .transform(test_df)
    .withColumn(
        "dataset_split",
        F.lit("test")
    )
)

all_predictions = (
    train_predictions
    .unionByName(test_predictions)
)

In [0]:
gold_predictions = (
    all_predictions

    .select(
        "order_id",
        "date_key",

        F.col("label")
         .alias("actual_delivery_delay_days"),

        F.round(
            F.col("predicted_delivery_delay_days"),
            2
        ).alias(
            "predicted_delivery_delay_days"
        ),

        F.round(
            F.col("label")
            - F.col("predicted_delivery_delay_days"),
            2
        ).alias(
            "prediction_error_days"
        ),

        F.round(
            F.abs(
                F.col("label")
                - F.col("predicted_delivery_delay_days")
            ),
            2
        ).alias(
            "absolute_error_days"
        ),

        (
            F.col("label") > 0
        ).alias("actual_late"),

        (
            F.col("predicted_delivery_delay_days") > 0
        ).alias("predicted_late"),

        "dataset_split"
    )

    .withColumn(
        "scored_at",
        F.current_timestamp()
    )
)

In [0]:
target_table = (
    f"{catalog_name}.{gold_schema}."
    "order_delivery_predictions"
)

(
    gold_predictions
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(
    f"Wrote {gold_predictions.count():,} "
    f"predictions to {target_table}"
)

In [0]:
predictions = spark.table(
    f"{catalog_name}.{gold_schema}."
    "order_delivery_predictions"
)

print(
    "Rows:",
    predictions.count()
)

print(
    "Distinct order_ids:",
    predictions
    .select("order_id")
    .distinct()
    .count()
)

print(
    "Null predictions:",
    predictions
    .filter(
        F.col(
            "predicted_delivery_delay_days"
        ).isNull()
    )
    .count()
)

predictions.show(
    10,
    truncate=False
)